# Distributed Systems Programming Fundamentals

{doc}`Spark Runtime Architecture <../01_apache_spark_runtime_architecture/02_spark_runtime_architecture>`
introduced the driver, which plans work, and the executors, which run it.
{doc}`Introduction to DataFrames <../02_spark_dataframes_and_sql/01_introduction_to_dataframes>`
showed that transformations only build a plan. This page connects the two:
how a plan is cut into units of parallel work, why some operations are much
more expensive than others, and how Spark copes when machines fail.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

trips = spark.read.table("samples.nyctaxi.trips")
lineitem = spark.read.table("samples.tpch.lineitem")

## Thinking Distributed

On a single machine, reading a value from memory is cheap. In a cluster,
getting a value that lives on *another* machine means serialising it, sending
it over the network, and deserialising it, which is far slower. Three rules of
thumb follow, and the rest of this page is mostly about applying them:

1. **Move computation to the data, not data to the computation.** The driver
   sends tasks to the executors, which hold the data
   ([Apache Spark: Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html)).
2. **Move as little data as possible between machines.** Filter early, select
   only the columns you need, and be deliberate about operations that
   redistribute data.
3. **Expect failure.** With enough machines, something is always failing.
   The system has to recover without your help.

## Partitions

A DataFrame's rows are split into **partitions**: chunks of rows that are
processed independently. The partition is the unit of parallelism: a single
task operates on a single partition
([Apache Spark: RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html#shuffle-operations)),
and a stage can run at most as many tasks at once as the cluster has slots
(CPU cores across all executors, as described in
{doc}`Spark Runtime Architecture <../01_apache_spark_runtime_architecture/02_spark_runtime_architecture>`).

Where do partitions come from?

- **Reading files**: Spark packs input files into partitions of up to
  `spark.sql.files.maxPartitionBytes`, 128 MB by default, also on serverless
  compute
  ([Set Spark configuration properties on Databricks](https://docs.databricks.com/aws/en/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs)).
  A 10 GB table is read as roughly 80 partitions.
- **After a shuffle**: the number of output partitions is set by
  `spark.sql.shuffle.partitions`. Open-source Spark defaults to 200
  ([Apache Spark: Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)).
  On serverless compute the default is `auto`
  ([Set Spark configuration properties on Databricks](https://docs.databricks.com/aws/en/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs)),
  which enables auto-optimised shuffle: Databricks determines the number
  itself
  ([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#enable-auto-optimized-shuffle)).
- **Explicitly**: `df.repartition(n)` or `df.coalesce(n)`, covered
  {ref}`below <repartition-vs-coalesce>`.

On classic compute with dedicated access mode you can check the count with
`df.rdd.getNumPartitions()`. On serverless the RDD APIs are not supported
([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#language-and-api-support)),
but the built-in `spark_partition_id` function shows how rows are spread out.
The small `samples.nyctaxi.trips` table fits in a single partition:

In [2]:
trips.groupBy(F.spark_partition_id().alias("partition")).count().show()

+---------+-----+
|partition|count|
+---------+-----+
|        0|21932|
+---------+-----+



The much larger TPC-H `lineitem` table, from the same `samples` catalog, is
read as several partitions:

In [3]:
(
    lineitem
    .groupBy(F.spark_partition_id().alias("partition"))
    .count()
    .orderBy("partition")
    .show()
)

+---------+-------+
|partition|  count|
+---------+-------+
|        0|3877133|
|        1|3633152|
|        2|3633152|
|        3|3633152|
|        4|3633152|
|        5|3633152|
|        6|4026518|
|        7|3930384|
+---------+-------+



Its roughly 30 million rows were read as 8 partitions of about 3.6 to 4
million rows each, so a stage reading it runs 8 tasks.

Partitions that are too **large** cause memory pressure and spill to disk;
spill happens when Spark runs low on execution memory and moves data to disk,
which can be expensive
([Skew and spill](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/long-spark-stage-page#spill)).
Partitions that are too **small** waste time on per-task overhead. After a
shuffle, AQE coalesces partitions towards an advisory size, 64 MB by default
in open-source Spark
([Apache Spark: Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html#coalescing-post-shuffle-partitions)),
so you rarely need to tune this by hand.

## Narrow and Wide Transformations

Transformations fall into two groups depending on whether a task needs data
from other partitions.

- **Narrow transformations** compute each output partition from exactly one
  input partition. `select`, `filter`, `withColumn`, and `union` are narrow.
  Each task can do its work locally, with no network traffic.
- **Wide transformations** need rows from *many* input partitions to compute
  one output partition. Aggregations by key, joins and repartitioning are the
  typical examples
  ([Apache Spark: RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html#shuffle-operations)),
  and so are `distinct` and `orderBy`. They require a **shuffle**.

```mermaid
:caption: Narrow transformations stay within a partition; wide transformations exchange data between all of them.
:align: center
:name: narrow-vs-wide

graph LR
    subgraph Narrow["Narrow: filter"]
        A1[P1] --> B1[P1']
        A2[P2] --> B2[P2']
        A3[P3] --> B3[P3']
    end
    subgraph Wide["Wide: groupBy"]
        C1[P1] --> D1[P1']
        C1 --> D2[P2']
        C2[P2] --> D1
        C2 --> D2
        C3[P3] --> D1
        C3 --> D2
    end
```

## Shuffles

A **shuffle** is Spark's mechanism for redistributing data so that it is
grouped differently across partitions. To aggregate by key, Spark must read
from all partitions to find all the values for each key, an all-to-all
operation
([Apache Spark: RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html#shuffle-operations)).
For `groupBy("l_shipmode")`, every row for one shipping mode must reach the
same task, wherever it started. A shuffle has two halves:

1. **Shuffle write**: the *map* tasks of the upstream stage sort their rows by
   target partition and write them to files on local disk.
2. **Shuffle read**: the *reduce* tasks of the downstream stage read the
   blocks that belong to them.

The shuffle involves disk I/O, data serialisation and network I/O, which
makes it expensive
([Apache Spark: RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html#performance-impact)).
Most Spark performance tuning comes down to shuffling less data, or shuffling
it less often.

Spark already limits the damage where it can. In a physical plan, shuffles
appear as exchange operators. Run an aggregation over `lineitem` and look at
the plan:

In [4]:
(
    lineitem
    .groupBy("l_shipmode")
    .agg(F.sum("l_extendedprice").alias("revenue"))
    .explain()
)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[l_shipmode#12499], functions=[finalmerge_sum(merge sum#12506, isEmpty#12507) AS sum(l_extendedprice)#12501])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#9717]
               +- PhotonShuffleExchangeSink hashpartitioning(l_shipmode#12499, 1024)
                  +- PhotonGroupingAgg(keys=[l_shipmode#12499], functions=[partial_sum(l_extendedprice#12490) AS (sum#12506, isEmpty#12507)])
                     +- PhotonScan parquet samples.tpch.lineitem[l_extendedprice#12490,l_shipmode#12499] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<l_extendedprice:decimal(18,2),l_shipmode:string>, R

Read the plan from the bottom up. On serverless compute Photon runs the
query, so the operators have Photon names. The first `PhotonGroupingAgg`
computes a `partial_sum` in each task, the `PhotonShuffleExchangeSink` /
`PhotonShuffleExchangeSource` pair is the shuffle, with
`hashpartitioning(l_shipmode, ...)` sending each key to one partition, and the
top `PhotonGroupingAgg` merges the partial sums (`finalmerge_sum`). Because of the partial step, only one row per key per
partition crosses the network, not every input row. The root
`AdaptiveSparkPlan` node shows that adaptive query execution is on, covered
{ref}`below <adaptive-query-execution>`.

## Jobs, Stages and Tasks

When an action runs, the driver turns the physical plan into a hierarchy of
work
([Apache Spark: Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html#glossary)):

- A **job** is the parallel computation spawned in response to an action.
- A job is divided into **stages** that depend on each other. Stage
  boundaries fall at shuffles: a stage cannot start reading until the stage
  that writes its shuffle files has finished. Within a stage, the narrow
  transformations all run inside the same task, one after another.
- A **task** is a unit of work sent to one executor, and a task operates on
  one partition, so a stage has one task per partition. All tasks in a stage
  run the same code on different data.

```mermaid
:caption: One action becomes one job, cut into stages at shuffle boundaries, each made of one task per partition.
:align: center

graph LR
    subgraph S1["Stage 1: read, filter, partial agg"]
        T1[Task 1]
        T2[Task 2]
        T3[Task 3]
    end
    subgraph S2["Stage 2: final agg"]
        U1[Task 1]
        U2[Task 2]
    end
    S1 -- shuffle --> S2
    S2 --> R[Result to driver]
```

When a stage has more tasks than the cluster has slots, the tasks run in
successive **waves**. With $T$ tasks and $S$ slots, a stage needs

$$
W = \left\lceil \frac{T}{S} \right\rceil
$$ (eq-task-waves)

waves. The last wave is often only partly full, leaving slots idle, which is
one reason partition counts that are a multiple of the slot count tend to
work well.

:::::{dropdown} Example: Counting Waves
:open:
:color: primary
:icon: pencil
:class-body: example-body

A classic cluster has 4 workers with 8 cores each, so $S = 32$ slots. A stage
reads a table that is split into $T = 1000$ partitions.

**Apply the wave formula, {eq}`eq-task-waves`.**

$$
W = \left\lceil \frac{1000}{32} \right\rceil = \lceil 31.25 \rceil = 32
$$

**Look at the last wave.**

The first 31 waves run $31 \times 32 = 992$ tasks, so the last wave runs only
$1000 - 992 = 8$ tasks while 24 slots sit idle.

The stage needs 32 waves, and the final one uses only a quarter of the
cluster. Rounding the partition count to $992$ or $1024$ would keep every
wave full.
:::::

On classic compute you can watch jobs, stages and tasks in the **Spark UI**.
On serverless compute the Spark UI is not available; use the query profile
instead
([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#ui-and-logging)).
In a notebook on serverless compute, the query profile is reached from the
link under the cell that ran the query, and it is also available from the
query history
([Query profile](https://docs.databricks.com/aws/en/sql/user/queries/query-profile#view-a-query-profile)).

(join-strategies)=
## Join Strategies

Joins are wide in general: to match keys, rows with the same key from both
sides must meet in the same task. Spark picks among a few strategies, which
you can also request with join hints
([Hints](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-hints#join-hint-types)):

- **Sort-merge join** (`MERGE` hint): shuffle *both* sides by the join key,
  sort each partition, and merge. Works for any size, but shuffles
  everything.
- **Broadcast hash join** (`BROADCAST` hint): if one side is small, send a
  full copy of it to every executor and build a hash table there. The large
  side is *not* shuffled at all. This is usually much faster.
- **Shuffle hash join** (`SHUFFLE_HASH` hint): shuffle both sides, then build
  a hash table per partition instead of sorting.

Open-source Spark broadcasts automatically when a table is smaller than
`spark.sql.autoBroadcastJoinThreshold`, 10 MB by default
([Apache Spark: Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html#automatically-broadcasting-joins)).
When you know better than the estimate, give a hint. `F.broadcast` is the
DataFrame form of the `BROADCAST` hint, which broadcasts the hinted side
regardless of the threshold
([Hints](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-hints#join-hint-types)):

In [5]:
zips = spark.createDataFrame(
    [(10001, "Chelsea"), (10003, "East Village"), (10023, "Upper West Side")],
    schema="zip INT, neighbourhood STRING",
)

trips_with_area = trips.join(
    F.broadcast(zips),
    trips["pickup_zip"] == zips["zip"],
    "left",
)
trips_with_area.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonBroadcastHashJoin [pickup_zip#12540], [zip#12528], LeftOuter, BuildRight, false, true, false
         :- PhotonScan parquet samples.nyctaxi.trips[tpep_pickup_datetime#12536,tpep_dropoff_datetime#12537,trip_distance#12538,fare_amount#12539,pickup_zip#12540,dropoff_zip#12541] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<tpep_pickup_datetime:timestamp,tpep_dropoff_datetime:timestamp,trip_distance:double,fare_a..., RequiredDataFilters: []
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#9760]
               +- PhotonShuffleExchangeSink SinglePartition
                  +- PhotonRowToColumnar
                

The physical plan shows a `PhotonBroadcastHashJoin` with `BuildRight`: the
small `zips` side goes through an exchange marked `EXECUTOR_BROADCAST`, while
the `trips` scan feeds the join directly, without a shuffle. Only broadcast data that comfortably fits in memory.

{doc}`DataFrame Relational Operations <../../02_developing_applications_with_apache_spark/03_dataframe_relational_operations>`
covers the join types themselves.

(adaptive-query-execution)=
## Adaptive Query Execution

Plans are chosen before execution, from estimates. **Adaptive Query
Execution (AQE)** uses runtime statistics to choose a more efficient plan,
and has been enabled by default since Apache Spark 3.2.0
([Apache Spark: Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution)).
On Databricks, AQE is enabled by default and
([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#capabilities)):

- **changes sort-merge joins into broadcast hash joins** when one side turns
  out to be small;
- **coalesces small partitions** into reasonably sized ones after a shuffle,
  so you don't have to tune `spark.sql.shuffle.partitions` by hand for every
  query;
- **handles skew** in sort-merge and shuffle hash joins by splitting
  oversized tasks;
- **detects and propagates empty relations**.

In `explain()` output, AQE plans have an `AdaptiveSparkPlan` node, usually as
the root, with an `isFinalPlan` flag that becomes `true` once execution
completes
([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#dataframeexplain)),
as in the plans above. On serverless compute AQE is not something you
configure: its settings are not among the few Spark properties serverless lets
you set
([Set Spark configuration properties on Databricks](https://docs.databricks.com/aws/en/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs)).

(data-skew)=
## Data Skew

A shuffle sends all rows for one key to one task. If one key is far more
common than the others, say 40% of all trips start in a single zip code,
that task gets far more data than the rest. The stage cannot finish until its
slowest task does, so the whole job waits on one overloaded core while the
others sit idle. This is **data skew**: one or just a few tasks take much
longer than the rest. In a stage's summary metrics, a maximum task duration
more than 50% above the 75th percentile suggests skew
([Skew and spill](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/long-spark-stage-page#skew)).

Remedies, roughly in order of preference:

1. **Let AQE handle it.** Databricks handles skew in joins by default through
   AQE, so skew join hints are not required
   ([Skew join optimization using skew hints](https://docs.databricks.com/aws/en/archive/legacy/skew-join)).
2. **Filter or pre-aggregate** before the shuffle so less data reaches the
   hot key.
3. **Broadcast** the smaller side of a join so the skewed side isn't
   shuffled at all, as in the {ref}`join example <join-strategies>` above.
4. **Salt the key**: add a random suffix to the hot key so its rows spread
   over several partitions, aggregate, then aggregate again without the salt.
   {doc}`Optimizing Apache Spark <../../04_monitoring_and_optimizing_spark_workloads_on_databricks/03_optimizing_apache_spark>`
   runs this on real data.

:::{admonition} Unverified: Salting as a Recommended Remedy
:class: warning

Salting is a general technique, demonstrated with real output in the page
linked above. It couldn't be confirmed that the official Databricks
documentation recommends it: the skew guidance on docs.databricks.com covers
AQE and skew hints, and links to remediation material outside the docs. Check
[Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe)
and the Spark UI guide's skew pages before relying on it as Databricks
guidance.
:::

(repartition-vs-coalesce)=
## Controlling Partitioning: `repartition` and `coalesce`

You rarely need to set partitioning by hand, but two methods let you:

- `df.repartition(n)` or `df.repartition(n, "col")` performs a **full
  shuffle** into exactly `n` partitions (hashed by `col` if given)
  ([Apache Spark API reference: DataFrame.repartition](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html)).
  Use it to increase parallelism, or to co-locate rows by key.
- `df.coalesce(n)` **merges** existing partitions down to `n` without a
  shuffle
  ([Apache Spark API reference: DataFrame.coalesce](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html)).
  It is cheap, but it can only reduce the count, and it can leave partitions
  unevenly sized.

The cell below repartitions the taxi data into 8 partitions by
`pickup_zip`, then counts the rows that ended up in each partition:

In [6]:
(
    trips
    .repartition(8, "pickup_zip")
    .groupBy(F.spark_partition_id().alias("partition"))
    .count()
    .orderBy("partition")
    .show()
)

+---------+-----+
|partition|count|
+---------+-----+
|        0| 3550|
|        1| 3199|
|        2| 1639|
|        3| 2185|
|        4| 2596|
|        5|  987|
|        6| 5199|
|        7| 2577|
+---------+-----+



Hash partitioning by `pickup_zip` sends every row of a zip code to the same
partition, so the partitions come out uneven: the largest holds over five
times as many rows as the smallest. That is a small taste of the data skew
described {ref}`above <data-skew>`.

Be careful with `coalesce(1)` and `repartition(1)`: a stage with a single
task keeps one CPU busy while the rest of the cluster sits idle
([One Spark task](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/one-spark-task)).

## Fault Tolerance

Spark does not replicate intermediate results to survive failures. Instead,
it keeps the **lineage** of every partition, the chain of transformations
that produced it, and rebuilds lost data from it
([Apache Spark: Research](https://spark.apache.org/research.html)). If a task
fails, Spark reruns it; only after 4 consecutive failures of the same task
(`spark.task.maxFailures`) does the job fail
([Apache Spark: Configuration](https://spark.apache.org/docs/latest/configuration.html#scheduling)).
Shuffle files written by completed upstream tasks are kept, so they don't
need to be recreated if the lineage is recomputed
([Apache Spark: RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html#performance-impact)).

This is why Spark's transformations should be **deterministic**: a retried
task recomputes its rows, and functions such as `rand()` without a seed, or a
UDF that calls an external service, can make the retry produce different rows
than the first attempt.

For stragglers, open-source Spark can relaunch slow tasks elsewhere
(**speculative execution**), but only when `spark.speculation` is turned on;
it is off by default
([Apache Spark: Configuration](https://spark.apache.org/docs/latest/configuration.html#scheduling)).

(driver-vs-executor-code)=
## Driver Code versus Executor Code

In a PySpark program, it matters where each line runs:

- Ordinary Python statements (loops, `if`s, variable assignments) run on the
  **driver**, or, with Databricks Connect, in your local Python process:
  general code runs locally, and DataFrame APIs run on Databricks compute
  ([What is Databricks Connect?](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/#how-does-it-work)).
- DataFrame operations are shipped to and run on the **executors**.
- A Python UDF's body also runs on the cluster, in a Python interpreter
  outside the JVM, which is why it is slower than built-in functions
  ([User-defined functions (UDFs)](https://docs.databricks.com/aws/en/udf/#performance-considerations)).

## Common Mistakes

- **Looping over rows in Python.** `for row in df.collect(): ...` pulls
  everything to one machine and processes it serially. Express the logic as
  DataFrame operations instead.
- **Building plans in large Python loops.** Each `withColumn` call adds a
  projection, so calling it in a loop to add many columns can generate big
  plans that cause performance issues and even a `StackOverflowException`.
  Use one `select` with all the new columns
  ([Apache Spark API reference: DataFrame.withColumn](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)):

  ```python
  # Slow: one projection per column
  for i in range(100):
      trips = trips.withColumn(f"fare_x{i}", F.col("fare_amount") * i)
  ```

  The cell below does the same with a single `select`:

In [7]:
wide = trips.select(
    "*",
    *[(F.col("fare_amount") * i).alias(f"fare_x{i}") for i in range(100)],
)
len(wide.columns)

106

- **Writing through `coalesce(1)` to get one file.** It forces the whole stage
  onto one task
  ([One Spark task](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/one-spark-task)).
  Write normally, and let readers consume the directory of part files.
- **Hand-tuning `spark.sql.shuffle.partitions` on serverless.** AQE coalesces
  partitions after shuffles, and the serverless default `auto` lets
  Databricks choose the number
  ([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#enable-auto-optimized-shuffle)).
- **Adding skew hints out of habit.** Databricks handles join skew by default
  through AQE
  ([Skew join optimization using skew hints](https://docs.databricks.com/aws/en/archive/legacy/skew-join)).

## A Short Checklist

When a Spark job is slower than expected, work through:

1. **Read less.** Filter on clustering columns and select only the columns
   you need, so Spark can skip files.
2. **Shuffle less.** Count the exchange operators in `explain()`. Can a join
   be broadcast? Can you aggregate before joining?
3. **Look for skew.** In the Spark UI or query profile, compare the slowest
   task in a stage with the 75th percentile.
4. **Size partitions sensibly.** Avoid thousands of tiny files and avoid a
   handful of huge partitions.
5. **Keep work off the driver.** No `collect()` on large data, no row loops,
   and prefer built-in functions to Python UDFs.

The {doc}`next section <../04_etl_with_the_dataframe_api/index>` applies
these ideas to a complete ETL pipeline.

## Resources

- [Performance tuning (Apache Spark)](https://spark.apache.org/docs/latest/sql-performance-tuning.html)
- [Diagnose cost and performance issues using the Spark UI (Databricks)](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/)
- [Set Spark configuration properties on Databricks (Databricks)](https://docs.databricks.com/aws/en/spark/conf)